# 9 · Transformations — Bronze → **Silver**

**Silver** is the cleaned, conformed, trustworthy layer: raw Bronze data that has
been **typed, standardized, de-duplicated, and enriched** (joined with reference
data). It's the "single source of truth" analysts and data scientists build on.

We'll turn `brewbox.orders_bronze` (from notebook 7) into `brewbox.orders_silver`,
showing the same logic in **both the DataFrame API and Spark SQL** — on Databricks
they compile to the same engine, so use whichever reads best.

In [ ]:
try:
    spark
except NameError:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()
from pyspark.sql import functions as F, Window
spark.sql("USE SCHEMA brewbox")

bronze = spark.table("brewbox.orders_bronze")
print("bronze rows:", bronze.count())
bronze.printSchema()

## 1 · Clean & type

Raw JSON gives loose types and inconsistent values. We cast `order_ts` to a real
timestamp, derive an `order_date`, standardize `status`, and add a boolean flag.

In [ ]:
typed = (bronze
    .withColumn("order_ts", F.to_timestamp("order_ts"))
    .withColumn("order_date", F.to_date("order_ts"))
    .withColumn("status", F.lower(F.trim("status")))
    .withColumn("is_completed", F.col("status") == "completed")
    .withColumn("amount", F.col("amount").cast("double")))
typed.select("order_id","order_ts","order_date","status","is_completed","amount").show(5)

## 2 · De-duplicate

Re-ingestion can create duplicates. Keep **one row per `order_id`** — the latest
by ingestion time — using a window + `row_number`. This makes the transform
**idempotent**.

In [ ]:
w = Window.partitionBy("order_id").orderBy(F.col("_ingested_at").desc())
deduped = (typed
    .withColumn("_rn", F.row_number().over(w))
    .filter("_rn = 1")
    .drop("_rn"))
print("rows before:", typed.count(), "-> after dedupe:", deduped.count())

## 3 · Enrich with reference data (joins)

Join the reference tables so Silver carries the descriptive context downstream
marts need: the customer's country & loyalty tier, and the store's region.

In [ ]:
customers = spark.table("brewbox.customers").select("customer_id","country","loyalty_tier")
stores    = spark.table("brewbox.stores").select("store_id","region")

silver = (deduped
    .join(customers, "customer_id", "left")
    .join(stores, "store_id", "left")
    .select("order_id","customer_id","store_id","country","loyalty_tier","region",
            "order_ts","order_date","status","is_completed","amount"))
silver.show(5)

## 4 · The same logic in Spark SQL

Register the inputs as temp views and express the whole transform as one SQL
statement. Same result, different style — pick whichever your team reads best (or
mix them).

In [ ]:
bronze.createOrReplaceTempView("orders_bronze_v")
result_sql = spark.sql("""
    WITH typed AS (
        SELECT order_id, customer_id, store_id,
               to_timestamp(order_ts) AS order_ts,
               to_date(to_timestamp(order_ts)) AS order_date,
               lower(trim(status)) AS status,
               CAST(amount AS DOUBLE) AS amount,
               _ingested_at,
               row_number() OVER (PARTITION BY order_id ORDER BY _ingested_at DESC) AS rn
        FROM orders_bronze_v
    )
    SELECT t.order_id, t.customer_id, t.store_id, c.country, c.loyalty_tier, s.region,
           t.order_ts, t.order_date, t.status, (t.status = 'completed') AS is_completed, t.amount
    FROM typed t
    LEFT JOIN brewbox.customers c ON c.customer_id = t.customer_id
    LEFT JOIN brewbox.stores    s ON s.store_id    = t.store_id
    WHERE t.rn = 1
""")
print("SQL version rows:", result_sql.count())

## 5 · Data-quality checks before you publish

Silver should be trustworthy — validate it before writing. Catch problems (nulls
in keys, negative amounts, unexpected statuses) so bad data never reaches Gold.

In [ ]:
checks = silver.agg(
    F.sum(F.col("order_id").isNull().cast("int")).alias("null_order_id"),
    F.sum((F.col("amount") < 0).cast("int")).alias("negative_amount"),
    F.countDistinct("status").alias("distinct_status"))
checks.show()
assert silver.filter("order_id IS NULL").count() == 0, "order_id must not be null"
print("data-quality checks passed")

## 6 · Write the Silver table

Persist as a managed Delta table. Using `overwrite` here keeps the transform
**idempotent** (re-running yields the same table). In production you'd often
`MERGE` incremental changes instead.

In [ ]:
silver.write.format("delta").mode("overwrite").saveAsTable("brewbox.orders_silver")
print("wrote brewbox.orders_silver:", spark.table("brewbox.orders_silver").count(), "rows")
spark.table("brewbox.orders_silver").show(5)

## 7 · pandas vs PySpark (when to use which)

pandas is great for **small** data that fits on the driver; PySpark scales the
*same* ideas across a cluster. On Databricks you can move between them, and even
run the **pandas API on Spark** to keep pandas syntax at scale.

In [ ]:
# Spark -> pandas (only when small enough for the driver)
pdf = spark.table("brewbox.orders_silver").limit(100).toPandas()
print("as pandas:", type(pdf).__name__, pdf.shape)

# pandas API on Spark: pandas syntax, distributed execution
psdf = spark.table("brewbox.orders_silver").pandas_api()
print(psdf.groupby("region")["amount"].sum().head())

## 8 · Exercises

**Exercise 1 —** From `brewbox.orders_silver`, compute completed revenue per
`region` (sum of `amount` where `is_completed`), sorted high to low.

In [ ]:
# Your turn (Exercise 1):

In [ ]:
# ✅ Solution 1
(spark.table("brewbox.orders_silver").filter("is_completed")
    .groupBy("region").agg(F.round(F.sum("amount"),2).alias("revenue"))
    .orderBy(F.desc("revenue")).show())

**Exercise 2 —** Are there any orders whose `country` is null after the join
(i.e., a customer_id with no matching customer)? Count them.

In [ ]:
# Your turn (Exercise 2):

In [ ]:
# ✅ Solution 2
print("orders with unmatched customer:",
      spark.table("brewbox.orders_silver").filter("country IS NULL").count())

**Exercise 3 —** Write the same "revenue by region" query in **Spark SQL** against
`brewbox.orders_silver`.

In [ ]:
# Your turn (Exercise 3):

In [ ]:
# ✅ Solution 3
spark.sql("""
    SELECT region, round(sum(amount),2) AS revenue
    FROM brewbox.orders_silver
    WHERE is_completed
    GROUP BY region ORDER BY revenue DESC
""").show()

## 9 · Recap & next

You cleaned, de-duplicated, and enriched Bronze into a trustworthy **Silver**
table (`brewbox.orders_silver`) — in both PySpark and SQL — with data-quality
checks and an idempotent write.

**Next → `10` Data modeling:** shape Silver into a **Gold** star schema
(dimensions + facts) that BI and ML consume. 🚀